# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
print("Connected!")

Connected!


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

base = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_total,
        SUM(gsc_clicks) AS gsc_clicks_total,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions_total,
        SUM(ga4_pageviews) AS ga4_pageviews_total,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_total,
        COUNT(*) AS days_observed,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS days_gsc_available
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

base["ctr"] = (base["gsc_clicks_total"] / base["gsc_impressions_total"]).fillna(0)
base["engagement_rate"] = (base["ga4_engaged_sessions_total"] / base["ga4_sessions_total"]).fillna(0)
base["gsc_coverage"] = (base["days_gsc_available"] / base["days_observed"]).fillna(0)
numeric_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total"]
base[numeric_cols] = base[numeric_cols].fillna(0)

label_data = con.sql(f"""
    WITH halves AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id, (second_half < first_half) AS is_declining
    FROM halves
""").df()

df = base.merge(label_data, on=["client_hash_id", "content_hash_id"])
feature_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total",
                 "days_observed", "days_gsc_available", "ctr", "engagement_rate", "gsc_coverage"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(
    train_df[feature_cols], train_df["is_declining"])
test_df["decline_probability"] = rf.predict_proba(test_df[feature_cols])[:, 1]
print("Model trained. Test rows:", len(test_df))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model trained. Test rows: 49823


In [3]:
p90 = test_df["decline_probability"].quantile(0.90)

def reason_code(row):
    if row["decline_probability"] >= p90 and row["gsc_avg_position"] <= 10:
        return "high_risk_ranked_page", "review_and_refresh"
    elif row["decline_probability"] >= p90:
        return "high_risk_low_visibility", "review_before_investing"
    elif row["gsc_coverage"] < 0.5:
        return "thin_data_coverage", "verify_data_before_acting"
    else:
        return "lower_priority", "monitor_only"

test_df[["reason_code", "action"]] = test_df.apply(lambda r: pd.Series(reason_code(r)), axis=1)
test_df["priority_score"] = test_df["decline_probability"] * np.log1p(test_df["gsc_impressions_total"])
ranked_queue = test_df.sort_values("priority_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

print("90th percentile:", round(p90, 3))
print(ranked_queue["reason_code"].value_counts())
ranked_queue[["rank", "content_hash_id", "decline_probability", "gsc_impressions_total", "reason_code", "action"]].head(10)

90th percentile: 0.469
reason_code
thin_data_coverage          29027
lower_priority              15808
high_risk_low_visibility     2701
high_risk_ranked_page        2287
Name: count, dtype: int64


,rank,content_hash_id,decline_probability,gsc_impressions_total,reason_code,action
0,1,content_761ad39548ba1d60,0.550463,77471.0,high_risk_low_visibility,review_before_investing
1,2,content_5941f92782343e72,0.549631,72724.0,high_risk_low_visibility,review_before_investing
2,3,content_dbf35ea262d38fb3,0.559125,47401.0,high_risk_low_visibility,review_before_investing
3,4,content_05220342facdcd7e,0.550758,47366.0,high_risk_low_visibility,review_before_investing
4,5,content_4caf70a28a8071db,0.553916,38898.0,high_risk_low_visibility,review_before_investing
5,6,content_0b7e2cd65fadec6f,0.526796,65264.0,high_risk_low_visibility,review_before_investing
6,7,content_04fd6dcd58ebdc88,0.537884,51598.0,high_risk_low_visibility,review_before_investing
7,8,content_83d539d00a7455d0,0.561664,29682.0,high_risk_low_visibility,review_before_investing
8,9,content_89c10d52fc81ac39,0.510136,81777.0,high_risk_low_visibility,review_before_investing
9,10,content_f3e85c887b94aa16,0.575703,12827.0,high_risk_low_visibility,review_before_investing


## Section 1: Ranked actions + reason codes

This queue uses the Random Forest model's predicted decline probability
(validated on a client-grouped split in ML-09) to rank content, paired with a
human-readable reason for each flag. Thresholds are set at the 90th percentile
of the observed probability distribution (not a fixed number), so they adapt
to the actual score range rather than assuming a fixed cutoff works.

**Reason codes:**
- `high_risk_ranked_page` — top-10% decline risk AND good search position
  (top 10) — the highest-value review candidate.
- `high_risk_low_visibility` — top-10% decline risk, but ranked below position
  10 — worth reviewing, lower visibility upside.
- `thin_data_coverage` — under 50% of the month has usable GSC data; the
  prediction itself is less reliable here.
- `lower_priority` — neither high risk nor thin coverage; monitor only.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use and limits

**Intended use:** This playbook is decision-support for a content team
deciding which pages to review first in a given month. It ranks and flags —
it does not make or execute editorial decisions.

**Who should use it:** Content strategists or SEO leads doing monthly
prioritization, not automated pipelines.

**Where it stops being valid:**
- This queue reflects March 2026 (month=2026-03) only — it is not validated
  on other months and should be regenerated, not reused, for a new month.
- The model was trained and grouped-split validated on this cohort of 104
  clients; performance on clients outside this warehouse is unknown.
- `thin_data_coverage` rows (29,027 of 49,823 — the majority) carry
  meaningfully less reliable predictions and should be treated as lower
  confidence, not ignored.
- This is observed, decision-support evidence from one month of data — not a
  causal claim that acting on a flagged page will improve its performance.

In [4]:
coverage_check = ranked_queue["gsc_coverage"].describe()
print("GSC coverage distribution across the queue:")
coverage_check


GSC coverage distribution across the queue:


,gsc_coverage
count,49823.000000
mean,0.400890
std,0.430765
min,0.000000
25%,0.000000
50%,0.142857
75%,0.935484
max,1.000000


**Data confirms the limit above:** median GSC coverage across the queue is only
14%, and 25% of rows have 0% coverage — meaning a large share of predictions
rest on very thin underlying data. The `thin_data_coverage` reason code (29,027
of 49,823 rows) is not a minor edge case; it describes a large portion of this
queue and should shape how confidently a reviewer acts on those rows.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review + the no-go list

**What a human must check before acting on any flagged row:**
1. `gsc_coverage` for that row — if below 0.5, treat the flag as a hint to
   investigate the data gap first, not an action-ready recommendation.
2. Whether the content was recently changed for reasons outside search
   performance (e.g. a planned rebrand, legal takedown, seasonal archive) —
   the model has no way to know this.
3. The actual page in a browser — a numeric flag never replaces reading the
   content and checking current SERP behavior directly.

**What should NEVER be automated:**
- Auto-publishing or auto-editing content based on `action` labels — every
  action here is a suggestion for a human review queue, not an execution
  command.
- Auto-deprioritizing or deleting content flagged `lower_priority` — absence
  of a decline signal is not evidence the page is safe to ignore long-term.
- Using this queue to make client-facing guarantees ("we will recover X% of
  traffic") — the underlying evidence is decision-support, not a guarantee
  (see claim-ladder in Section 2).
- Applying this exact model/thresholds to a different month or a different
  client cohort without re-validation — the honest-split gap found in ML-09
  (precision@50 dropping from 0.84 to 0.56) shows performance does not
  transfer automatically to unseen groups.

In [6]:
high_confidence_actionable = ranked_queue[
    (ranked_queue["reason_code"].isin(["high_risk_ranked_page", "high_risk_low_visibility"])) &
    (ranked_queue["gsc_coverage"] >= 0.5)
]
print("High-risk AND reliable-coverage rows (safest to prioritize first):", len(high_confidence_actionable))
print("High-risk but thin-coverage rows (flag, but verify data first):",
      len(ranked_queue[(ranked_queue["reason_code"].isin(["high_risk_ranked_page", "high_risk_low_visibility"])) &
                        (ranked_queue["gsc_coverage"] < 0.5)]))


High-risk AND reliable-coverage rows (safest to prioritize first): 4988
High-risk but thin-coverage rows (flag, but verify data first): 0


**Confirms the review priority:** all 4,988 high-risk flagged rows
(`high_risk_ranked_page` + `high_risk_low_visibility`) already have reliable
GSC coverage (≥50%) — none of the highest-priority flags overlap with the
thin-data-coverage group. This means a reviewer can trust the top of the queue
without an extra coverage check, though the `thin_data_coverage` group
(29,027 rows) still needs the data-gap investigation described above before
any of those rows are actioned.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / retrain triggers

**Signs the recommendations have gone stale:**
1. **New month arrives** — this queue is built from month=2026-03 only. Each
   new month needs a freshly regenerated queue, not a reused one.
2. **Reason-code distribution shifts sharply** — if `thin_data_coverage`
   grows well beyond ~58% of the queue (its March level), a data pipeline
   issue (not a content issue) is more likely than a real change in client
   behavior.
3. **Precision@50 on a fresh grouped-split check drops well below the ML-09
   honest baseline (0.56)** — re-validate on the newest month before trusting
   the queue further.
4. **New clients join the cohort** — since ML-09 showed a real gap between
   same-client and unseen-client performance (0.84 vs 0.56), a growing share
   of new/unseen clients in the queue is a reason to re-check grouped
   precision@50 specifically on that new-client subset.

**Retrain trigger:** retrain when either (a) three consecutive months show
grouped-split precision@50 below 0.45 (20% relative drop from the ML-09
baseline), or (b) the client cohort grows by more than 20% since the last
training run.

In [7]:
current_thin_pct = (ranked_queue["reason_code"] == "thin_data_coverage").mean()
print(f"Current thin_data_coverage share: {current_thin_pct:.1%}")
print("Monitor this percentage each month — a sharp rise signals a data pipeline issue, not content decline.")


Current thin_data_coverage share: 58.3%
Monitor this percentage each month — a sharp rise signals a data pipeline issue, not content decline.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



This section writes the final ranked queue and key metrics to `work/outputs/`
(queue CSV, kept out of git by design — regenerated on every run) and a
metrics JSON (committed — the receipt these numbers trace back to).

In [8]:
import os
import json

# --- Export 1: the ranked queue CSV (stays out of git, regenerated each run) ---
os.makedirs("work/outputs", exist_ok=True)
export_cols = ["rank", "content_hash_id", "client_hash_id", "decline_probability",
               "gsc_impressions_total", "gsc_avg_position", "gsc_coverage",
               "reason_code", "action"]
ranked_queue[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("Queue exported:", len(ranked_queue), "rows")

# --- Export 2: metrics JSON (this one IS committed — the receipts) ---
metrics = {
    "month": MONTH,
    "n_rows_scored": int(len(ranked_queue)),
    "reason_code_distribution": ranked_queue["reason_code"].value_counts().to_dict(),
    "p90_threshold": round(float(p90), 3),
    "median_gsc_coverage": round(float(ranked_queue["gsc_coverage"].median()), 3),
    "thin_data_coverage_share": round(float(current_thin_pct), 3),
    "honest_split_precision_at_50": 0.56,   # from ML-09
    "honest_split_auc": 0.804,              # from ML-09
    "random_split_precision_at_50_for_comparison": 0.84,  # from ML-09, shown as the "before" caution
    "model": "RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)",
    "split_design": "GroupShuffleSplit by client_hash_id, test_size=0.3, random_state=42"
}

with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics saved:")
print(json.dumps(metrics, indent=2))


Queue exported: 49823 rows

Metrics saved:
{
  "month": "2026-03",
  "n_rows_scored": 49823,
  "reason_code_distribution": {
    "thin_data_coverage": 29027,
    "lower_priority": 15808,
    "high_risk_low_visibility": 2701,
    "high_risk_ranked_page": 2287
  },
  "p90_threshold": 0.469,
  "median_gsc_coverage": 0.143,
  "thin_data_coverage_share": 0.583,
  "honest_split_precision_at_50": 0.56,
  "honest_split_auc": 0.804,
  "random_split_precision_at_50_for_comparison": 0.84,
  "model": "RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)",
  "split_design": "GroupShuffleSplit by client_hash_id, test_size=0.3, random_state=42"
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.